
# ScentSync — AI-Powered Fragrance Recommendation & Dupe Discovery System

## Project Motivation

Fragrance enthusiasts often struggle to:
- Discover fragrances with a specific scent profile
- Find affordable alternatives (“dupes”) to expensive designer fragrances
- Search fragrances using natural language descriptions such as:
  - “warm cozy date night scent”
  - “fresh aquatic summer fragrance”
  - “cool gym scent”

The goal of ScentSync was to build an AI-powered semantic fragrance recommendation engine capable of:
1. Understanding fragrance notes and accords
2. Recommending similar fragrances
3. Finding affordable clone alternatives
4. Comparing traditional NLP retrieval models against transformer-based semantic embeddings

This project was intentionally designed as an NLP + recommendation systems project to demonstrate:
- Semantic search
- Vector similarity
- NLP preprocessing
- Recommendation pipelines
- Transformer embeddings
- Model comparison
- AI-powered retrieval systems



# Tech Stack

## Languages & Libraries
- Python
- Pandas
- NumPy
- Scikit-learn
- Sentence-Transformers
- Streamlit
- Pickle

## NLP / ML Techniques
- TF-IDF Vectorization
- Sentence-BERT Embeddings
- Cosine Similarity
- Semantic Search
- Top-K Retrieval

## Dataset
- Fragrantica Kaggle Dataset
- 24,000+ fragrance entries



# Step 1 — Import Libraries

We first import the libraries required for:
- data preprocessing
- NLP vectorization
- semantic embeddings
- cosine similarity search
- saving/loading trained models


In [ ]:

import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer



# Step 2 — Load Dataset

The fragrance dataset contains:
- fragrance names
- brands
- top notes
- middle notes
- base notes
- accords

The dataset uses semicolon separators (`;`) instead of commas, so `sep=";"` is required.

We also use:
```python
encoding="latin1"
```
because the dataset contains special characters that caused UTF-8 decoding errors.


In [ ]:

df = pd.read_csv(
    "DataSet/fra_cleaned.csv",
    sep=";",
    encoding="latin1"
)

print(df.shape)

df.head()



# Step 3 — Keep Relevant Columns

We only keep the columns useful for recommendation retrieval.

These include:
- fragrance name
- brand
- scent notes
- accords

This reduces unnecessary noise in the NLP pipeline.


In [ ]:

df = df[[
    "Perfume",
    "Brand",
    "Gender",
    "Top",
    "Middle",
    "Base",
    "mainaccord1",
    "mainaccord2",
    "mainaccord3"
]]

df = df.fillna("")

df.head()



# Step 4 — Feature Engineering

## Why feature engineering matters

Raw fragrance data is not directly usable by NLP models.

We therefore create TWO separate text representations:

---

## 1. `combined` (for TF-IDF)

This is a keyword-based representation:
```text
vanilla, amber, woody, musk
```

TF-IDF performs very well on sparse keyword-heavy data because it relies heavily on exact word overlap.

---

## 2. `bert_text` (for Sentence-BERT)

This is a natural-language representation:
```text
This fragrance has top notes of vanilla...
```

Transformer models such as Sentence-BERT perform much better when text resembles natural language sentences instead of raw tags.

This was one of the most important discoveries during the project:
- TF-IDF surprisingly performed very strongly on sparse fragrance metadata
- Sentence-BERT improved contextual understanding after converting notes into natural-language descriptions


In [ ]:

# TF-IDF representation
df["combined"] = (
    df["Top"] + ", " +
    df["Middle"] + ", " +
    df["Base"] + ", " +
    df["mainaccord1"] + ", " +
    df["mainaccord2"] + ", " +
    df["mainaccord3"]
)

# Sentence-BERT representation
df["bert_text"] = (
    "This fragrance has top notes of " + df["Top"] +
    ", middle notes of " + df["Middle"] +
    ", base notes of " + df["Base"] +
    ". Main accords include " +
    df["mainaccord1"] + ", " +
    df["mainaccord2"] + ", " +
    df["mainaccord3"] + "."
)

df["combined"] = (
    df["combined"]
    .str.lower()
    .str.replace("unknown", "")
)

df = df.drop_duplicates(
    subset=["Perfume", "Brand"]
)

df[["combined", "bert_text"]].head()



# Step 5 — Build TF-IDF Model

## Why TF-IDF?

TF-IDF was implemented as the baseline recommendation model.

The goal was to:
- create a fast keyword-based retrieval system
- establish a baseline for comparison against Sentence-BERT

TF-IDF works well for:
- exact note overlap
- accord matching
- structured fragrance metadata

The model converts fragrance text into numerical vectors.


In [ ]:

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    df["combined"]
)

print(tfidf_matrix.shape)



# Step 6 — Save TF-IDF Model

The trained vectorizer and matrix are saved using Pickle.

This prevents rebuilding the model every time the application runs.


In [ ]:

pickle.dump(
    vectorizer,
    open("Model/tfidf_vectorizer.pkl", "wb")
)

pickle.dump(
    tfidf_matrix,
    open("Model/tfidf_matrix.pkl", "wb")
)

pickle.dump(
    df,
    open("Model/data.pkl", "wb")
)

print("TF-IDF model saved successfully!")



# Step 7 — TF-IDF Search Function

## Why cosine similarity?

Cosine similarity measures how similar two vectors are.

A similarity score closer to:
- `1.0` → very similar
- `0.0` → unrelated

The system retrieves the:
```text
Top 5 most similar fragrances
```

based on cosine similarity.


In [ ]:

def search_tfidf(query):

    query_vec = vectorizer.transform([query])

    sim = cosine_similarity(
        query_vec,
        tfidf_matrix
    ).flatten()

    top_indices = sim.argsort()[-5:][::-1]

    results = df.iloc[top_indices].copy()

    results["similarity"] = sim[top_indices]

    return results[[
        "Perfume",
        "Brand",
        "similarity"
    ]]



# Step 8 — Build Sentence-BERT Embeddings

## Why add Sentence-BERT?

TF-IDF relies heavily on exact word overlap.

This creates limitations for queries like:
- “cozy winter fragrance”
- “luxury masculine scent”
- “summer vacation perfume”

We therefore implemented Sentence-BERT to:
- understand semantic meaning
- capture contextual fragrance descriptions
- improve natural-language search

We used:
```text
all-MiniLM-L6-v2
```

because it provides:
- good semantic quality
- relatively fast inference
- lightweight transformer embeddings


In [ ]:

bert_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

bert_embeddings = bert_model.encode(
    df["bert_text"].tolist(),
    show_progress_bar=True
)

bert_embeddings = np.array(
    bert_embeddings
)

print(bert_embeddings.shape)



# Step 9 — Save BERT Embeddings

BERT embeddings are expensive to generate.

Saving them prevents recomputing embeddings every time the application starts.


In [ ]:

pickle.dump(
    bert_embeddings,
    open("Model/bert_embeddings.pkl", "wb")
)

print("BERT embeddings saved successfully!")



# Step 10 — Sentence-BERT Semantic Search

This search pipeline:
1. Converts the query into a semantic embedding
2. Computes cosine similarity against all fragrance embeddings
3. Returns the top semantic matches

This allows contextual fragrance search instead of simple keyword matching.


In [ ]:

def search_bert(query):

    query_vec = bert_model.encode([query])

    sim = cosine_similarity(
        query_vec,
        bert_embeddings
    ).flatten()

    top_indices = sim.argsort()[-5:][::-1]

    results = df.iloc[top_indices].copy()

    results["similarity"] = sim[top_indices]

    return results[[
        "Perfume",
        "Brand",
        "similarity"
    ]]



# Step 11 — Compare TF-IDF vs Sentence-BERT

One major goal of the project was to compare:
- traditional NLP retrieval
- transformer-based semantic retrieval

This helped evaluate:
- keyword overlap performance
- semantic understanding
- contextual recommendation quality


In [ ]:

queries = [
    "warm cozy date night",
    "fresh aquatic summer",
    "sweet vanilla sexy",
    "cool gym fresh"
]

for q in queries:

    print("\nQUERY:", q)

    print("\nTF-IDF RESULTS")
    print(search_tfidf(q))

    print("\nBERT RESULTS")
    print(search_bert(q))



# Key Observation

One surprising result from the project:

## TF-IDF performed extremely well on fragrance metadata

Why?
Because fragrance notes are highly keyword-oriented.

Example:
```text
vanilla, amber, musk
```

This structure naturally favors TF-IDF.

However:
- Sentence-BERT performed better on mood-based and contextual queries
- BERT improved semantic understanding after feature engineering

This became an important machine learning insight from the project:
> Simpler models can outperform transformer models when the dataset structure strongly favors keyword overlap.



# Step 12 — Dupe Detection System

## Goal

Find affordable fragrances that smell similar to expensive fragrances.

Examples:
- Baccarat Rouge 540 → Lattafa dupes
- Creed Aventus → Armaf alternatives

The dupe system:
1. Retrieves top 50 semantic matches internally
2. Filters known clone-house brands
3. Returns the best affordable alternatives

The top-50 retrieval strategy was important because:
- true dupes may not appear in the top 5 recommendations
- internal retrieval and external display are different recommendation objectives


In [ ]:

dupe_brands = [
    "armaf",
    "lattafa-perfumes",
    "lattafa",
    "afnan",
    "rasasi",
    "maison-alhambra"
]

def search_bert_extended(query):

    query_vec = bert_model.encode([query])

    sim = cosine_similarity(
        query_vec,
        bert_embeddings
    ).flatten()

    top_indices = sim.argsort()[-50:][::-1]

    results = df.iloc[top_indices].copy()

    results["similarity"] = sim[top_indices]

    return results

def get_dupes(query):

    results = search_bert_extended(query)

    dupes = results[
        results["Brand"]
        .str.lower()
        .isin(dupe_brands)
    ]

    return dupes.head(5)



# Step 13 — Test Dupe Detection


In [ ]:

query = "Baccarat Rouge 540"

print("TOP DUPES")
print(get_dupes(query))



# Step 14 — Streamlit Web Application

The final stage of the project was building an interactive Streamlit application.

The app includes:
- real-time fragrance search
- TF-IDF vs BERT selection
- top recommendations
- affordable alternatives

This transformed the project from:
```text
ML scripts
```

into:
```text
a deployable AI-powered product
```


In [ ]:

# Run in terminal:

# streamlit run Notebook/app.py



# Final Project Summary

## What ScentSync Demonstrates

- NLP preprocessing
- Semantic search
- Recommendation systems
- Sentence-BERT embeddings
- Vector similarity search
- Hybrid recommendation pipelines
- Feature engineering
- Retrieval systems
- AI-powered web applications

## Future Improvements
- FAISS vector indexing
- Recommendation reranking
- Fragrance image integration
- User personalization
- Cloud deployment
- Real-time inference optimization
